# Chunking et embedding

## Tokenizer

Le chunking est l'opération de découper en plus petits morceaux un texte avant de l'envoyre dans l'embedder.

EN général ~500 tokens mais cette taille ne constitue pas une règle absolue.

Les modèles d'embeddings acceptent souvent bien plus mais alors l'information est noyée dans un vecteur trop ambigu.

La taille et la manière de construire les chunks peut avoir une grande influence par la suite sur le retrieval.

In [75]:
import numpy as np
from scipy.spatial import distance

from pathlib import Path
from transformers import AutoTokenizer

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.line_chunker import LineBasedTokenChunker
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

from openai import OpenAI

In [41]:
CORPUS = Path('data/veltis')
EMBEDDER_PATH = Path('../models/embedder/bge-m3')


tokenizer = AutoTokenizer.from_pretrained(EMBEDDER_PATH)
print(tokenizer.model_max_length)

text = "Le chunking est l'opération de découper en plus petits morceaux un texte avant de l'envoyre dans l'embedder."

tokenizer.tokenize(text)[:3]

8192


['▁Le', '▁chun', 'king']

### Chunking naïf

Le principal problème des méthodes naïves est qu'elle découpe les documents n'impôrte sans faire attention à la structure du document. Des informations qui devraient rester dans un même chunk peuvent séparés dans deux chunks distincts


In [42]:
converter = DocumentConverter()

doc = converter.convert(CORPUS / 'intranet_faq_rh.html').document

docling_tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=64)
naive_chunker = LineBasedTokenChunker(tokenizer=docling_tokenizer)

chunks = naive_chunker.chunk(dl_doc=doc)

for idx, chunk in enumerate(chunks):
    print(f'Chunk {idx+1}:\n {chunk.text}\n--------')

Chunk 1:
 # FAQ Ressources humaines

Les réponses aux questions les plus fréquentes. Mise à jour : 2 février 2026.

## Congés et absences

### Combien de jours de congé ai-je par an ?


--------
Chunk 2:
 Chaque personne dispose de 20 jours de congé légaux par an. Un jour supplémentaire est accordé par tranche de cinq années d'ancienneté, avec un maximum de quatre jours supplémentaires.

### Comment demander un congé ?


--------
Chunk 3:
 Les congés se demandent dans le portail MonVeltis au moins deux semaines à l'avance. Votre manager valide la demande dans le portail.

### Que faire en cas de maladie ?


--------
Chunk 4:
 Prévenez votre manager avant 9 h le premier jour d'absence et envoyez le certificat médical au service RH dans les 48 heures.

## Rémunération

### Quand le salaire est-il versé ?


--------
Chunk 5:
 Le salaire est versé le dernier jour ouvrable du mois. La fiche de paie est disponible dans MonVeltis.

### Où trouver la grille des salaires ?

La grille des rémuné

### Chunking Hiérarchique

Le chunking hiérarchique découpe les documents en prenant en compte la structure des documents. Les chunks sont constitués par élément (titre, sous-titre, paragraphe, etc.).

Avec ce découpage, les questions réponses ne sont plus tronquées

In [43]:
hierarchical_chunker = HierarchicalChunker(tokenizer=docling_tokenizer)

chunks = hierarchical_chunker.chunk(dl_doc=doc)

for idx, chunk in enumerate(chunks):
    print(chunk.meta.headings)
    print(f'Chunk {idx+1}:\n {chunk.text}\n--------')

['FAQ Ressources humaines']
Chunk 1:
 Les réponses aux questions les plus fréquentes. Mise à jour : 2 février 2026.
--------
['FAQ Ressources humaines', 'Congés et absences', 'Combien de jours de congé ai-je par an ?']
Chunk 2:
 Chaque personne dispose de 20 jours de congé légaux par an. Un jour supplémentaire est accordé par tranche de cinq années d'ancienneté, avec un maximum de quatre jours supplémentaires.
--------
['FAQ Ressources humaines', 'Congés et absences', 'Comment demander un congé ?']
Chunk 3:
 Les congés se demandent dans le portail MonVeltis au moins deux semaines à l'avance. Votre manager valide la demande dans le portail.
--------
['FAQ Ressources humaines', 'Congés et absences', 'Que faire en cas de maladie ?']
Chunk 4:
 Prévenez votre manager avant 9 h le premier jour d'absence et envoyez le certificat médical au service RH dans les 48 heures.
--------
['FAQ Ressources humaines', 'Rémunération', 'Quand le salaire est-il versé ?']
Chunk 5:
 Le salaire est versé le de

La méthode ``contextualize()`` permet de renvoyer le texte enrichi qui contient donc les entêtes ainsi que des infos sur le document.

C'est ce texte qui sera envoyer dans l'embedder.

In [44]:
chunks = hierarchical_chunker.chunk(dl_doc=doc)
for chunk in chunks:
    print(hierarchical_chunker.contextualize(chunk))
    print('-----')

FAQ Ressources humaines
Les réponses aux questions les plus fréquentes. Mise à jour : 2 février 2026.
-----
FAQ Ressources humaines
Congés et absences
Combien de jours de congé ai-je par an ?
Chaque personne dispose de 20 jours de congé légaux par an. Un jour supplémentaire est accordé par tranche de cinq années d'ancienneté, avec un maximum de quatre jours supplémentaires.
-----
FAQ Ressources humaines
Congés et absences
Comment demander un congé ?
Les congés se demandent dans le portail MonVeltis au moins deux semaines à l'avance. Votre manager valide la demande dans le portail.
-----
FAQ Ressources humaines
Congés et absences
Que faire en cas de maladie ?
Prévenez votre manager avant 9 h le premier jour d'absence et envoyez le certificat médical au service RH dans les 48 heures.
-----
FAQ Ressources humaines
Rémunération
Quand le salaire est-il versé ?
Le salaire est versé le dernier jour ouvrable du mois. La fiche de paie est disponible dans MonVeltis.
-----
FAQ Ressources humaines

### Chunking Hybride

Le chunking hiérarchique a cependant à problème est qu'il ne soucie pas de la taille des chunks. Si le document n'est pas bien structuré, il peut y avoir une grande variance dans la taille des chunks.

Des chunks trop courts avec peu d'information ou bien des chunks trop longs avec trop d'informations.

Le chunking hybride part du découpage hiérarchique mais y ajoute donc deux éléments:
- Les chunks trop longs sont redécoupés en cherchant des entêtes d'un niveau inférieur
- Les chunks trop courts sont regroupés en recherhant des entêtes d'un niveau supérieur

C'est le tokenizer à choisir par défaut.

In [45]:
hybrid_chunker = HybridChunker(tokenizer=docling_tokenizer)

chunks = hybrid_chunker.chunk(doc)

for chunk in chunks:
    print(hybrid_chunker.contextualize(chunk))
    print('-----')


FAQ Ressources humaines
Les réponses aux questions les plus fréquentes. Mise à jour : 2 février 2026.
-----
FAQ Ressources humaines
Congés et absences
Combien de jours de congé ai-je par an ?
Chaque personne dispose de 20 jours de congé légaux par an.
-----
FAQ Ressources humaines
Congés et absences
Combien de jours de congé ai-je par an ?
Un jour supplémentaire est accordé par tranche de cinq années d'ancienneté, avec un maximum de quatre jours supplémentaires.
-----
FAQ Ressources humaines
Congés et absences
Comment demander un congé ?
Les congés se demandent dans le portail MonVeltis au moins deux semaines à l'avance. Votre manager valide la demande dans le portail.
-----
FAQ Ressources humaines
Congés et absences
Que faire en cas de maladie ?
Prévenez votre manager avant 9 h le premier jour d'absence et envoyez le certificat médical au service RH dans les 48 heures.
-----
FAQ Ressources humaines
Rémunération
Quand le salaire est-il versé ?
Le salaire est versé le dernier jour ouvra

In [46]:
docling_tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer, max_tokens=512)
hybrid_chunker = HybridChunker(tokenizer=docling_tokenizer)

chunks = hybrid_chunker.chunk(doc)

for chunk in chunks:
    print(hybrid_chunker.contextualize(chunk))
    print('-----')


FAQ Ressources humaines
Les réponses aux questions les plus fréquentes. Mise à jour : 2 février 2026.
-----
FAQ Ressources humaines
Congés et absences
Combien de jours de congé ai-je par an ?
Chaque personne dispose de 20 jours de congé légaux par an. Un jour supplémentaire est accordé par tranche de cinq années d'ancienneté, avec un maximum de quatre jours supplémentaires.
-----
FAQ Ressources humaines
Congés et absences
Comment demander un congé ?
Les congés se demandent dans le portail MonVeltis au moins deux semaines à l'avance. Votre manager valide la demande dans le portail.
-----
FAQ Ressources humaines
Congés et absences
Que faire en cas de maladie ?
Prévenez votre manager avant 9 h le premier jour d'absence et envoyez le certificat médical au service RH dans les 48 heures.
-----
FAQ Ressources humaines
Rémunération
Quand le salaire est-il versé ?
Le salaire est versé le dernier jour ouvrable du mois. La fiche de paie est disponible dans MonVeltis.
-----
FAQ Ressources humaines

## Embedding

L'embedding est l'étape qui permet numériser nos chunks et tokens.

In [97]:
BASE_URL = 'http://100.85.184.14:8001/v1'
client = OpenAI(
    base_url=BASE_URL,
    api_key='no-sk'
)

sentences = [
    'A combien de congé ai-je droit par an ?',
    'How many days off I get per year ?',
    'Vous avez droits à 20 jours de congé par année en plus de jours fériés.',
    "Il n'y avait pas de peur, parce que qu'il n'y avait pas de mort."
]

embeddings = []
for sentence in sentences:
    r = client.embeddings.create(model='bge-m3', input=sentence)
    embeddings.append(np.array([v.embedding for v in r.data]))
embeddings


[array([[-0.05729844,  0.00293674, -0.04656925, ..., -0.0106531 ,
         -0.04299286, -0.02408361]], shape=(1, 1024)),
 array([[-0.06007103, -0.01080557, -0.05324046, ...,  0.00257095,
         -0.07604696, -0.00549766]], shape=(1, 1024)),
 array([[-0.04516516,  0.02807818, -0.03792523, ..., -0.0073806 ,
         -0.07352468, -0.00391303]], shape=(1, 1024)),
 array([[ 0.0155851 , -0.0053889 , -0.05328922, ...,  0.00547069,
          0.01031434, -0.01531247]], shape=(1, 1024))]

In [98]:
print(distance.cosine(
    embeddings[0].ravel()
    , embeddings[1].ravel()
))

print(
distance.cosine(
    embeddings[0].ravel()
    , embeddings[2].ravel()
))

print(
distance.cosine(
    embeddings[0].ravel()
    , embeddings[3].ravel()
))

0.11343568873706056
0.24075974369052877
0.7169038351318239
